# LeetCode 398 — Random Pick Index bằng Python

Link đề: https://leetcode.com/problems/random-pick-index/ · Độ khó: **Medium** · Chủ đề: Reservoir Sampling, Hash Table, Math, Randomized

Skeleton LeetCode:

```python
class Solution:

    def __init__(self, nums: List[int]):
        ...

    def pick(self, target: int) -> int:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(nums)
# param_1 = obj.pick(target)
```

Ghi chú: đề dịch sang tiếng Việt từ đề gốc. Bài này có **tính ngẫu nhiên**: ta không so với đáp án cứng mà kiểm tra bằng **tính chất** (chỉ số trả về phải hợp lệ) và **thống kê với seed cố định** (phép thử `chi-bình-phương`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề thiết kế lớp: khởi tạo với mảng `nums` có thể **trùng lặp**, mỗi lần `pick(target)` phải trả về **một chỉ số `i` ngẫu nhiên đều** trong các chỉ số thỏa `nums[i] == target`.
2. Nắm công cụ Python: mô-đun `random` (`random.seed`, `random.randint`, `random.choice`), `dict` với `setdefault`/`defaultdict`, `enumerate`, `bisect`, `collections.Counter`.
3. Đi từ cách **quét mảng + gom danh sách chỉ số mỗi lần** → **tiền xử lý bảng băm `giá trị → danh sách chỉ số`** → **reservoir sampling (lấy mẫu bể chứa) `O(1)` bộ nhớ thêm** → **mảng đã sắp xếp + `bisect`**.
4. Hiểu mấu chốt reservoir sampling: ở lần khớp thứ `n`, **thay đáp án bằng chỉ số hiện tại với xác suất `1/n`**; chứng minh mọi chỉ số cuối cùng cùng có xác suất `1/k`.
5. Biết cách **kiểm thử chương trình ngẫu nhiên**: cố định `seed`, đếm tần suất, so với kỳ vọng bằng `chi-bình-phương`, và bắt được cài đặt thiên lệch.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` có thể chứa **phần tử trùng nhau**. Hãy cài đặt lớp `Solution`:

- `Solution(int[] nums)`: khởi tạo đối tượng với mảng `nums`.
- `int pick(int target)`: chọn ngẫu nhiên một chỉ số `i` sao cho `nums[i] == target`. Nếu có nhiều chỉ số hợp lệ thì **mỗi chỉ số phải có xác suất bằng nhau**. Đảm bảo `target` luôn xuất hiện trong `nums`.

**Ví dụ:**

```
Input
["Solution", "pick", "pick", "pick"]
[[[1, 2, 3, 3, 3]], [3], [1], [3]]
Output
[null, 4, 0, 2]      (một kết quả có thể xảy ra)
```

- `new Solution([1, 2, 3, 3, 3])`
- `pick(3)` trả về `2`, `3` hoặc `4` (mỗi chỉ số xác suất `1/3`).
- `pick(1)` luôn trả về `0` (chỉ `nums[0] == 1`).
- `pick(3)` lại trả về `2`, `3` hoặc `4`.

**Ràng buộc:** `1 <= nums.length <= 2 * 10^4`; `-2^31 <= nums[i] <= 2^31 - 1`; `target` là một số có trong `nums`; tối đa `10^4` lần gọi `pick`.

In [1]:
import bisect
import random
import time
from collections import Counter, defaultdict
from itertools import product
from fractions import Fraction
from typing import List

nums_demo = [1, 2, 3, 3, 3]
print("các chỉ số của giá trị 3:", [i for i, v in enumerate(nums_demo) if v == 3])
print("các chỉ số của giá trị 1:", [i for i, v in enumerate(nums_demo) if v == 1])

các chỉ số của giá trị 3: [2, 3, 4]
các chỉ số của giá trị 1: [0]


### 2.1. Đề "thiết kế lớp" chạy thế nào? Bộ chạy chuỗi lệnh

Trên LeetCode, hệ thống nhận hai danh sách: tên thao tác (`ops`) và tham số (`args`), tạo đối tượng ở thao tác đầu rồi gọi lần lượt các phương thức. Ta tự dựng một **bộ chạy** `run_ops` mô phỏng điều đó; kết quả của constructor là `None` (ứng với `null`).

In [2]:
def run_ops(cls, ops, args):
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == "Solution":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


# Mô phỏng ví dụ đề (kết quả pick(3) là ngẫu nhiên nên ở đây ta chỉ chờ đúng "họ" các giá trị hợp lệ)
class _DemoFirst:
    """Lớp giả tối thiểu, chỉ để minh họa cách bộ chạy hoạt động (luôn trả chỉ số đầu tiên)."""
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        return self.nums.index(target)


print(run_ops(_DemoFirst, ["Solution", "pick", "pick", "pick"], [[[1, 2, 3, 3, 3]], [3], [1], [3]]))

[None, 2, 0, 2]


## 3. Kiến thức nền

### 3.1. Mô-đun `random`

- `random.randint(a, b)` trả về số nguyên **ngẫu nhiên đều** trong `[a, b]` (**gồm cả hai đầu**).
- `random.choice(seq)` chọn một phần tử ngẫu nhiên đều từ dãy không rỗng.
- `random.seed(x)` đặt "hạt giống": cùng `seed` thì dãy số ngẫu nhiên **lặp lại y hệt**. Nhờ vậy ta kiểm thử được chương trình ngẫu nhiên một cách **tái lập** (reproducible).

In [3]:
random.seed(398)
a = [random.randint(1, 6) for _ in range(8)]
random.seed(398)
b = [random.randint(1, 6) for _ in range(8)]
print(a)
print(b)
print("cùng seed ⇒ cùng dãy:", a == b)
print("randint(1, 1) luôn là:", {random.randint(1, 1) for _ in range(20)})

[4, 4, 2, 5, 4, 3, 4, 6]
[4, 4, 2, 5, 4, 3, 4, 6]
cùng seed ⇒ cùng dãy: True
randint(1, 1) luôn là: {1}


### 3.2. `dict` gom nhóm: `giá trị → danh sách chỉ số`

Khi cần tra "những chỉ số nào có giá trị `v`?", ta gom một lần vào `dict`: `pos[v]` là danh sách các chỉ số. `defaultdict(list)` tự tạo danh sách rỗng cho khóa chưa có.

In [4]:
nums = [1, 2, 3, 3, 3]
pos = defaultdict(list)
for i, v in enumerate(nums):
    pos[v].append(i)
print(dict(pos))                 # {1: [0], 2: [1], 3: [2, 3, 4]}
print(pos[3], len(pos[3]))

{1: [0], 2: [1], 3: [2, 3, 4]}
[2, 3, 4] 3


### 3.3. `Counter` để đếm tần suất

`Counter(iterable)` đếm số lần xuất hiện của từng phần tử. Ta sẽ dùng nó để **đếm xem mỗi chỉ số được chọn bao nhiêu lần** qua nhiều lần `pick`.

In [5]:
c = Counter(["a", "b", "a", "c", "a"])
print(c, c["a"], c["z"])         # Counter không có khóa -> 0, không lỗi
print(sorted(c.items()))

Counter({'a': 3, 'b': 1, 'c': 1}) 3 0
[('a', 3), ('b', 1), ('c', 1)]


### 3.4. `bisect`: tìm đoạn các phần tử bằng nhau trong mảng đã sắp xếp

Với mảng đã sắp xếp, `bisect_left(arr, x)` cho vị trí đầu tiên `>= x`, `bisect_right(arr, x)` cho vị trí đầu tiên `> x`. Đoạn `[bisect_left, bisect_right)` chứa đúng tất cả các phần tử bằng `x`.

In [6]:
arr = [1, 2, 3, 3, 3, 7]
print(bisect.bisect_left(arr, 3), bisect.bisect_right(arr, 3))     # 2 5 -> đoạn [2, 5)
pairs = sorted((v, i) for i, v in enumerate([3, 1, 3, 2, 3]))      # sắp theo (giá trị, chỉ số)
print(pairs)
lo = bisect.bisect_left(pairs, (3, -1))
hi = bisect.bisect_left(pairs, (4, -1))
print("các chỉ số của giá trị 3:", [pairs[k][1] for k in range(lo, hi)])

2 5
[(1, 1), (2, 3), (3, 0), (3, 2), (3, 4)]
các chỉ số của giá trị 3: [0, 2, 4]


### 3.5. Chi-bình-phương: đo mức "lệch khỏi đều"

Nếu `k` kết quả đều xác suất như nhau và ta thử `T` lần thì kỳ vọng mỗi kết quả là `E = T / k`. Thống kê

`chi2 = Σ (đếm_i − E)² / E`

nhỏ nghĩa là gần đều. Với `k − 1` bậc tự do, mức tới hạn `0.1%` (hầu như chỉ vượt khi thật sự **lệch**): `k=2: 10.83`, `k=3: 13.82`, `k=4: 16.27`, `k=5: 18.47`, `k=6: 20.52`, `k=7: 22.46`, `k=8: 24.32`, `k=9: 26.12`, `k=10: 27.88`. Ta hardcode các giá trị này (với `k` lớn hơn bảng, code dùng công thức xấp xỉ Wilson-Hilferty).

In [7]:
class Chi2Table(dict):
    """Bảng ngưỡng 0.1% theo SỐ KẾT QUẢ k (bậc tự do k - 1); k ngoài bảng dùng xấp xỉ Wilson-Hilferty."""
    def __missing__(self, k):
        df = k - 1
        z = 3.0902                                   # phân vị 99.9% của phân phối chuẩn
        h = 2 / (9 * df)
        return df * (1 - h + z * h ** 0.5) ** 3


CHI2_CRIT = Chi2Table({2: 10.83, 3: 13.82, 4: 16.27, 5: 18.47, 6: 20.52, 7: 22.46, 8: 24.32, 9: 26.12, 10: 27.88})


def chi_square(counts):
    total = sum(counts)
    expected = total / len(counts)
    return sum((c - expected) ** 2 / expected for c in counts)


def looks_uniform(counts):
    return chi_square(counts) < CHI2_CRIT[len(counts)]


print(chi_square([3333, 3334, 3333]), looks_uniform([3333, 3334, 3333]))     # rất đều
print(chi_square([5000, 3000, 2000]), looks_uniform([5000, 3000, 2000]))     # lệch rõ

0.00019999999999999998 True
1399.9999999999998 False


## 4. Ý tưởng nền tảng: tính tay

Mảng `[1, 2, 3, 3, 3]`, `target = 3`.

1. Các chỉ số hợp lệ là `{2, 3, 4}` — `k = 3` ứng viên.
2. Để chọn đều, ta chỉ cần chọn một số ngẫu nhiên đều trong `{0, 1, 2}` rồi lấy chỉ số thứ đó trong danh sách `[2, 3, 4]`.
3. Với `target = 1` chỉ có một ứng viên `{0}` nên luôn trả `0`.

Vậy bài toán quy về: **liệt kê các chỉ số hợp lệ rồi chọn đều**. Phần "chọn đều" là `random.choice` hoặc `random.randint`. Câu hỏi thiết kế còn lại là: **liệt kê lúc nào (mỗi lần pick hay chỉ một lần) và có phải lưu cả danh sách không?** Đó là điểm khác nhau giữa các cách.

## 5. Cách 1 — Brute Force: mỗi lần `pick` quét mảng, gom danh sách rồi chọn

**Ý tưởng:** không tiền xử lý. Mỗi lần `pick(target)`: duyệt `nums`, ghi lại các chỉ số có `nums[i] == target` vào danh sách `candidates`, rồi `random.choice(candidates)`.

In [8]:
class SolutionScan:
    def __init__(self, nums: List[int]):
        self.nums = nums

    def pick(self, target: int) -> int:
        candidates = [i for i, v in enumerate(self.nums) if v == target]
        return random.choice(candidates)


random.seed(1)
s = SolutionScan([1, 2, 3, 3, 3])
print([s.pick(3) for _ in range(10)], "| pick(1) =", s.pick(1))
assert all(i in (2, 3, 4) for i in [s.pick(3) for _ in range(100)])

[2, 4, 2, 3, 2, 3, 3, 3, 4, 3] | pick(1) = 0


**Độ phức tạp:** mỗi lần `pick`: Time `O(n)`, Space `O(k)` (danh sách `candidates`; tệ nhất `O(n)` nếu mọi phần tử bằng nhau). `__init__` chỉ `O(1)`.

**Nút thắt:** với `n = 2 * 10^4` và `10^4` lần `pick` ta tốn tới `2 * 10^8` bước, và **mỗi lần** còn cấp phát danh sách tạm. Hai hướng cải thiện: (1) tiền xử lý một lần để `pick` rất nhanh (đánh đổi bộ nhớ) — Cách 2; (2) không cần danh sách, chỉ cần **chọn ngay trong lúc quét** — Cách 3 (reservoir).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá của reservoir sampling:** ta không cần biết trước có bao nhiêu ứng viên `k`. Duyệt từng ứng viên theo thứ tự; giữ **một đáp án hiện tại** `ans`. Khi gặp ứng viên thứ `n` (đếm từ 1), **thay `ans` bằng ứng viên này với xác suất `1/n`**, ngược lại giữ nguyên.

**Vì sao mọi ứng viên đều có xác suất cuối cùng `1/k`?** Xét ứng viên thứ `i` trong `k` ứng viên. Nó là đáp án cuối khi:

1. Được chọn lúc gặp: xác suất `1/i`;
2. Không bị thay ở các lần gặp thứ `i+1, ..., k`: lần thứ `j` giữ nguyên với xác suất `(j-1)/j`.

Nhân lại: `(1/i) * (i/(i+1)) * ((i+1)/(i+2)) * ... * ((k-1)/k) = 1/k` (tích **thu gọn dần**). Ta kiểm bằng phân số **chính xác** (module `fractions`).

In [9]:
def final_probability(i, k):
    """Xác suất ứng viên thứ i (1-based) là đáp án cuối khi có k ứng viên."""
    p = Fraction(1, i)
    for j in range(i + 1, k + 1):
        p *= Fraction(j - 1, j)
    return p


for k in [1, 2, 3, 4, 5, 10]:
    probs = [final_probability(i, k) for i in range(1, k + 1)]
    print(f"k={k:>2}: {[str(p) for p in probs]} | tổng = {sum(probs)}")
    assert all(p == Fraction(1, k) for p in probs)

k= 1: ['1'] | tổng = 1
k= 2: ['1/2', '1/2'] | tổng = 1
k= 3: ['1/3', '1/3', '1/3'] | tổng = 1
k= 4: ['1/4', '1/4', '1/4', '1/4'] | tổng = 1
k= 5: ['1/5', '1/5', '1/5', '1/5', '1/5'] | tổng = 1
k=10: ['1/10', '1/10', '1/10', '1/10', '1/10', '1/10', '1/10', '1/10', '1/10', '1/10'] | tổng = 1


Thấy được rồi: với mọi `k`, mọi `i`, xác suất đều bằng `1/k`. Có thể kiểm chứng thêm theo kiểu **vét cạn mọi kết quả ngẫu nhiên**: tại lần gặp thứ `j`, "rút" `x_j` đều trong `1..j` và thay khi `x_j == j`. Có `1 * 2 * ... * k = k!` tổ hợp `(x_1, ..., x_k)` **đồng khả năng**; đếm xem ứng viên nào thắng ở mỗi tổ hợp.

In [10]:
def enumerate_reservoir(k):
    wins = Counter()
    for xs in product(*[range(1, j + 1) for j in range(1, k + 1)]):
        ans = 0
        for j, x in enumerate(xs, start=1):
            if x == j:                 # thay với xác suất 1/j
                ans = j
        wins[ans] += 1
    return wins


for k in [3, 4, 5, 6]:
    wins = enumerate_reservoir(k)
    print(f"k={k}: tổng tổ hợp = {sum(wins.values())}, số lần thắng của từng ứng viên = {[wins[i] for i in range(1, k + 1)]}")
    assert len(set(wins.values())) == 1          # tất cả bằng nhau

k=3: tổng tổ hợp = 6, số lần thắng của từng ứng viên = [2, 2, 2]
k=4: tổng tổ hợp = 24, số lần thắng của từng ứng viên = [6, 6, 6, 6]
k=5: tổng tổ hợp = 120, số lần thắng của từng ứng viên = [24, 24, 24, 24, 24]
k=6: tổng tổ hợp = 720, số lần thắng của từng ứng viên = [120, 120, 120, 120, 120, 120]


## 7. Các cách tiếp theo

### Cách 2 — Bảng băm tiền xử lý `giá trị → danh sách chỉ số`

**Giải thích từng đoạn:**

1. Trong `__init__`: duyệt `nums` một lần, gom chỉ số theo giá trị vào `self.pos`.
2. `pick(target)`: `random.choice(self.pos[target])` — chọn đều trong danh sách đã lưu sẵn.
3. Đánh đổi: bộ nhớ `O(n)` để `pick` chỉ còn `O(1)`.

In [11]:
class SolutionMap:
    def __init__(self, nums: List[int]):
        self.pos = defaultdict(list)
        for i, v in enumerate(nums):
            self.pos[v].append(i)

    def pick(self, target: int) -> int:
        return random.choice(self.pos[target])


random.seed(2)
s = SolutionMap([1, 2, 3, 3, 3])
print([s.pick(3) for _ in range(10)], "| pick(1) =", s.pick(1))

[2, 2, 2, 3, 2, 4, 4, 3, 3, 4] | pick(1) = 0


**Độ phức tạp Cách 2:** `__init__` Time `O(n)`, `pick` Time `O(1)`; Space `O(n)`.

Nếu đề nhấn mạnh "mảng rất lớn, bộ nhớ hạn chế" thì Cách 2 không phù hợp; ta chuyển sang Cách 3.

### Cách 3 — Reservoir sampling (chốt)

**Giải thích từng đoạn:**

1. `__init__` chỉ giữ tham chiếu tới `nums` (không sao chép, không tạo cấu trúc phụ).
2. `pick`: `n = 0` (số ứng viên đã gặp), `ans = -1`. Duyệt `nums`; mỗi khi `nums[i] == target`: `n += 1`, rút `x = random.randint(1, n)`, nếu `x == n` thì `ans = i` (xác suất `1/n`).
3. Trả về `ans`. Lần khớp đầu tiên `n = 1` ⇒ `randint(1, 1) = 1 == n` ⇒ luôn nhận ứng viên đầu, nên `ans` luôn hợp lệ.

In [12]:
class SolutionReservoir:
    def __init__(self, nums: List[int]):
        self.nums = nums

    def pick(self, target: int) -> int:
        n = 0
        ans = -1
        for i, v in enumerate(self.nums):
            if v == target:
                n += 1                                  # ứng viên thứ n
                if random.randint(1, n) == n:           # xác suất 1/n
                    ans = i
        return ans


random.seed(3)
s = SolutionReservoir([1, 2, 3, 3, 3])
print([s.pick(3) for _ in range(10)], "| pick(1) =", s.pick(1))

[2, 4, 3, 4, 3, 4, 4, 2, 3, 4] | pick(1) = 0


**Độ phức tạp Cách 3:** `__init__` `O(1)`; `pick` Time `O(n)`; Space **`O(1)`** thêm. Đây là lời giải "chuẩn" của bài (câu hỏi follow-up của đề: không dùng thêm bộ nhớ).

### Cách 4 — Sắp xếp cặp `(giá trị, chỉ số)` + `bisect`

**Giải thích từng đoạn:**

1. `__init__`: tạo danh sách cặp `(nums[i], i)` rồi **sắp xếp**. Các cặp cùng giá trị nằm liền nhau, đã sắp theo chỉ số.
2. `pick(target)`: dùng `bisect_left` tìm `lo` (cặp đầu của `target`) và `bisect_left` của `target + 1` để tìm `hi` (cặp ngay sau nhóm). Đoạn `[lo, hi)` chính là toàn bộ ứng viên.
3. Chọn `j = random.randrange(lo, hi)` rồi trả `pairs[j][1]`. Mỗi `pick` chỉ `O(log n)`.

In [13]:
class SolutionBisect:
    def __init__(self, nums: List[int]):
        self.pairs = sorted((v, i) for i, v in enumerate(nums))

    def pick(self, target: int) -> int:
        lo = bisect.bisect_left(self.pairs, (target, -1))          # cặp (target, i) đầu tiên
        hi = bisect.bisect_left(self.pairs, (target + 1, -1))      # cặp đầu tiên của giá trị lớn hơn
        return self.pairs[random.randrange(lo, hi)][1]


random.seed(4)
s = SolutionBisect([1, 2, 3, 3, 3])
print([s.pick(3) for _ in range(10)], "| pick(1) =", s.pick(1))

[2, 3, 2, 4, 3, 3, 2, 2, 2, 2] | pick(1) = 0


**Độ phức tạp Cách 4:** `__init__` Time `O(n log n)`; `pick` Time `O(log n)`; Space `O(n)`. Dùng khi cần `pick` rất nhanh nhưng không muốn dựng `dict` (hoặc khi giá trị cần truy vấn theo khoảng).

**Nhận xét:** Cách 2 và 4 đều "chọn đều trên danh sách ứng viên đã lưu"; Cách 3 chọn "ngay trong luồng". Chúng đều đúng, khác nhau về cách cân bằng **thời gian ↔ bộ nhớ**.

## 8. TRACE từng bước

Trace reservoir với `nums = [1, 2, 3, 3, 3]`, `target = 3`. Để dễ quan sát ta **giả lập** các số ngẫu nhiên `x` đã định trước (thay vì rút thật): lần gặp thứ 2 rút `x = 2`, lần thứ 3 rút `x = 1`.

| `i` | `nums[i]` | khớp? | `n` | `x = randint(1, n)` | `x == n`? | `ans` |
|---|---|---|---|---|---|---|
| 0 | 1 | không | 0 | — | — | -1 |
| 1 | 2 | không | 0 | — | — | -1 |
| 2 | 3 | có | 1 | 1 (luôn) | có | 2 |
| 3 | 3 | có | 2 | 2 | có | 3 |
| 4 | 3 | có | 3 | 1 | không | 3 |

Kết quả: `3`. Chỉ số `2` đã bị `3` thay ở lần thứ hai, còn `4` không thay được vì `x = 1 != 3`.

In [14]:
def trace_reservoir(nums, target, draws=None):
    """draws: danh sách các giá trị x định trước cho từng lần khớp (None = rút ngẫu nhiên thật)."""
    n, ans = 0, -1
    print(f"{'i':>2} | {'nums[i]':>7} | khớp | {'n':>2} | {'x':>3} | x==n | ans")
    for i, v in enumerate(nums):
        if v != target:
            print(f"{i:>2} | {v:>7} | không | {n:>2} | {'—':>3} | —    | {ans}")
            continue
        n += 1
        x = draws[n - 1] if draws is not None else random.randint(1, n)
        take = (x == n)
        if take:
            ans = i
        print(f"{i:>2} | {v:>7} | có   | {n:>2} | {x:>3} | {'có' if take else 'không':<4} | {ans}")
    return ans


print("kết quả:", trace_reservoir([1, 2, 3, 3, 3], 3, draws=[1, 2, 1]))

 i | nums[i] | khớp |  n |   x | x==n | ans
 0 |       1 | không |  0 |   — | —    | -1
 1 |       2 | không |  0 |   — | —    | -1
 2 |       3 | có   |  1 |   1 | có   | 2
 3 |       3 | có   |  2 |   2 | có   | 3
 4 |       3 | có   |  3 |   1 | không | 3
kết quả: 3


In [15]:
# Trace thứ hai: seed cố định, rút thật; mảng dài hơn, nhiều ứng viên
random.seed(398)
res = trace_reservoir([5, 3, 5, 5, 1, 5, 5], 5)
print("kết quả:", res, "(luôn là một trong các chỉ số của giá trị 5: 0, 2, 3, 5, 6)")
assert res in (0, 2, 3, 5, 6)

 i | nums[i] | khớp |  n |   x | x==n | ans
 0 |       5 | có   |  1 |   1 | có   | 0
 1 |       3 | không |  1 |   — | —    | 0
 2 |       5 | có   |  2 |   2 | có   | 2
 3 |       5 | có   |  3 |   1 | không | 2
 4 |       1 | không |  3 |   — | —    | 2
 5 |       5 | có   |  4 |   4 | có   | 5
 6 |       5 | có   |  5 |   3 | không | 5
kết quả: 5 (luôn là một trong các chỉ số của giá trị 5: 0, 2, 3, 5, 6)


## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 3** (reservoir sampling): không dùng thêm bộ nhớ. Nhớ `import random` ở đầu file khi nộp.

In [16]:
import random


class Solution:

    def __init__(self, nums: List[int]):
        self.nums = nums

    def pick(self, target: int) -> int:
        n = 0
        ans = 0
        for i, v in enumerate(self.nums):
            if v == target:
                n += 1                              # đây là ứng viên thứ n
                if random.randint(1, n) == n:       # thay đáp án với xác suất 1/n
                    ans = i
        return ans


# Your Solution object will be instantiated and called as such:
# obj = Solution(nums)
# param_1 = obj.pick(target)

random.seed(398)
out = run_ops(Solution, ["Solution", "pick", "pick", "pick"], [[[1, 2, 3, 3, 3]], [3], [1], [3]])
print(out)
assert out[0] is None and out[1] in (2, 3, 4) and out[2] == 0 and out[3] in (2, 3, 4)

[None, 3, 0, 4]


## 10. Vì sao đúng

1. **Hợp lệ:** `ans` chỉ được gán khi `nums[i] == target`. Ứng viên đầu tiên có `n = 1` nên `randint(1, 1) == 1` luôn thỏa, vậy sau khi gặp ứng viên đầu `ans` luôn là chỉ số hợp lệ. (Đề đảm bảo `target` có trong `nums`.)
2. **Đều:** mục 6 chứng minh xác suất ứng viên thứ `i` là đáp án cuối bằng `(1/i) * Π_{j=i+1..k} (j-1)/j = 1/k`; mục 6 cũng kiểm bằng phân số chính xác và vét cạn `k!` tổ hợp.
3. **Độc lập giữa các lần `pick`:** mỗi lần `pick` dùng lại các số ngẫu nhiên mới, không giữ trạng thái giữa các lần gọi.
4. **Không cần biết `k` trước:** chỉ cần biết "đây là ứng viên thứ mấy" tại thời điểm gặp.

## 11. Time / Space Complexity

| Cách | `__init__` | `pick` | Space |
|---|---|---|---|
| 1. Quét + danh sách | `O(1)` | `O(n)` | `O(k)` mỗi lần |
| 2. Bảng băm | `O(n)` | `O(1)` | `O(n)` |
| 3. Reservoir | `O(1)` | `O(n)` | **`O(1)`** thêm |
| 4. Sắp xếp + bisect | `O(n log n)` | `O(log n)` | `O(n)` |

Với giới hạn `n = 2 * 10^4`, `10^4` lần `pick`: Cách 3 cỡ `2 * 10^8` phép (đạt trên LeetCode vì test thực tế thường nhẹ hơn), Cách 2 chỉ `10^4` phép.

## 12. Bộ test

Vì bài ngẫu nhiên, ta **không** so đáp án cứng. Dùng: (a) **tính hợp lệ** — chỉ số trả về luôn có giá trị bằng `target`; (b) **thống kê với seed cố định** — đếm tần suất rồi chi-bình-phương; (c) **tính chất chính xác** — phân số/vét cạn ở mục 6; (d) **bắt cài đặt sai** — thống kê phải *bác bỏ* các cài đặt thiên lệch.

In [17]:
# 12.1 Hợp lệ trên ví dụ đề + biên: chỉ một phần tử, mọi phần tử bằng nhau, số âm và cực trị int32
ALL_CLASSES = [Solution, SolutionScan, SolutionMap, SolutionReservoir, SolutionBisect]
random.seed(12)
for cls in ALL_CLASSES:
    out = run_ops(cls, ["Solution", "pick", "pick", "pick"], [[[1, 2, 3, 3, 3]], [3], [1], [3]])
    assert out[0] is None and out[1] in (2, 3, 4) and out[2] == 0 and out[3] in (2, 3, 4)
    assert cls([7]).pick(7) == 0                              # n = 1
    assert cls([-2**31, 2**31 - 1, -2**31]).pick(2**31 - 1) == 1
    assert cls([-2**31, 2**31 - 1, -2**31]).pick(-2**31) in (0, 2)
    assert cls([5, 5, 5, 5]).pick(5) in (0, 1, 2, 3)
print("OK: ví dụ đề + biên cho 5 lớp")

OK: ví dụ đề + biên cho 5 lớp


In [18]:
# 12.2 Hợp lệ trên mảng ngẫu nhiên đúng ràng buộc (seed cố định): chỉ số luôn trỏ tới đúng giá trị
random.seed(2024)
for _ in range(300):
    n = random.randint(1, 60)
    nums = [random.choice([-2**31, -1, 0, 1, 2, 5, 2**31 - 1]) for _ in range(n)]
    for cls in ALL_CLASSES:
        obj = cls(nums)
        for target in set(nums):
            for _ in range(5):
                i = obj.pick(target)
                assert 0 <= i < n and nums[i] == target, (cls.__name__, nums, target, i)
print("OK: 300 mảng ngẫu nhiên, mọi pick đều trỏ tới đúng giá trị")

OK: 300 mảng ngẫu nhiên, mọi pick đều trỏ tới đúng giá trị


In [19]:
# 12.3 Thống kê: ví dụ [1,2,3,3,3], pick(3) 30000 lần, seed cố định, chi-bình-phương cho 3 chỉ số
def sample_counts(cls, nums, target, trials):
    obj = cls(nums)
    cnt = Counter(obj.pick(target) for _ in range(trials))
    return cnt


for cls in ALL_CLASSES:
    random.seed(398)
    cnt = sample_counts(cls, [1, 2, 3, 3, 3], 3, 30000)
    counts = [cnt[i] for i in (2, 3, 4)]
    chi = chi_square(counts)
    print(f"{cls.__name__:<18} đếm={counts} chi2={chi:5.2f} (ngưỡng {CHI2_CRIT[3]})")
    assert set(cnt) <= {2, 3, 4} and looks_uniform(counts)

Solution           đếm=[9940, 9973, 10087] chi2= 1.19 (ngưỡng 13.82)
SolutionScan       đếm=[9917, 9956, 10127] chi2= 2.50 (ngưỡng 13.82)
SolutionMap        đếm=[9917, 9956, 10127] chi2= 2.50 (ngưỡng 13.82)


SolutionReservoir  đếm=[9940, 9973, 10087] chi2= 1.19 (ngưỡng 13.82)


SolutionBisect     đếm=[9917, 9956, 10127] chi2= 2.50 (ngưỡng 13.82)


In [20]:
# 12.4 Thống kê với k khác nhau (2..9 ứng viên, xen lẫn nhiễu), chỉ chạy Reservoir và Solution (lời giải nộp)
random.seed(99)
for k in range(2, 10):
    nums = []
    for _ in range(k):
        nums.extend([random.choice([100, 200, 300]) for _ in range(random.randint(0, 3))])
        nums.append(7)
    positions = [i for i, v in enumerate(nums) if v == 7]
    obj = Solution(nums)
    cnt = Counter(obj.pick(7) for _ in range(4000 * k))
    counts = [cnt[i] for i in positions]
    ok = looks_uniform(counts)
    print(f"k={k}: n={len(nums):>2}, chi2={chi_square(counts):6.2f} (ngưỡng {CHI2_CRIT[k]}) -> {'đều' if ok else 'LỆCH'}")
    assert ok

k=2: n= 6, chi2=  1.06 (ngưỡng 10.83) -> đều
k=3: n= 9, chi2=  1.13 (ngưỡng 13.82) -> đều


k=4: n= 8, chi2=  8.30 (ngưỡng 16.27) -> đều
k=5: n=14, chi2=  1.48 (ngưỡng 18.47) -> đều


k=6: n=16, chi2=  3.84 (ngưỡng 20.52) -> đều


k=7: n=19, chi2=  2.40 (ngưỡng 22.46) -> đều


k=8: n=16, chi2=  4.99 (ngưỡng 24.32) -> đều


k=9: n=26, chi2= 19.82 (ngưỡng 26.12) -> đều


In [21]:
# 12.5 Kiểm tra theo giá trị: mảng nhiều giá trị, mỗi giá trị đều phải đều trên các chỉ số của chính nó
random.seed(555)
nums = [random.choice([1, 2, 3]) for _ in range(40)]
obj = Solution(nums)
for target in (1, 2, 3):
    positions = [i for i, v in enumerate(nums) if v == target]
    cnt = Counter(obj.pick(target) for _ in range(1500 * len(positions)))
    counts = [cnt[i] for i in positions]
    if len(positions) >= 2:
        assert looks_uniform(counts), (target, counts)
    print(f"target={target}: {len(positions)} ứng viên, chi2 = {chi_square(counts) if len(positions) > 1 else 0:.2f}")

target=1: 15 ứng viên, chi2 = 10.73
target=2: 11 ứng viên, chi2 = 4.30


target=3: 14 ứng viên, chi2 = 11.02


In [22]:
# 12.6 Cài đặt SAI phải bị thống kê bác bỏ: (a) thay với xác suất 1/2; (b) luôn thay (trả chỉ số cuối)
class WrongHalf:
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        ans = -1
        for i, v in enumerate(self.nums):
            if v == target and (ans == -1 or random.randint(0, 1) == 0):    # SAI: luôn 1/2
                ans = i
        return ans


class WrongAlwaysLast:
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        ans = -1
        for i, v in enumerate(self.nums):
            if v == target:
                ans = i                                                      # SAI: luôn lấy cái cuối
        return ans


for cls in (WrongHalf, WrongAlwaysLast):
    random.seed(398)
    cnt = sample_counts(cls, [1, 2, 3, 3, 3], 3, 30000)
    counts = [cnt[i] for i in (2, 3, 4)]
    print(f"{cls.__name__:<16} đếm={counts} chi2={chi_square(counts):10.2f} -> {'đều' if looks_uniform(counts) else 'BỊ BÁC BỎ (lệch)'}")
    assert not looks_uniform(counts)

WrongHalf        đếm=[7399, 7555, 15046] chi2=   3820.53 -> BỊ BÁC BỎ (lệch)


WrongAlwaysLast  đếm=[0, 0, 30000] chi2=  60000.00 -> BỊ BÁC BỎ (lệch)


In [23]:
# 12.7 Cùng seed -> cùng kết quả (tái lập), seed khác -> thường khác
def run_series(seed):
    random.seed(seed)
    obj = Solution([1, 2, 3, 3, 3, 3, 3, 3])
    return [obj.pick(3) for _ in range(20)]


assert run_series(5) == run_series(5)
assert run_series(5) != run_series(6)
print("OK: tái lập theo seed")

OK: tái lập theo seed


In [24]:
# 12.8 Hiệu năng: n = 20000; Cách 2 chạy đủ 10^4 lần pick, Cách 3 chạy 500 lần (mỗi lần O(n))
random.seed(20000)
big = [random.randint(-10, 10) for _ in range(20000)]
t = time.perf_counter()
m = SolutionMap(big)
for _ in range(10**4):
    m.pick(random.choice(big))
t_map = time.perf_counter() - t

t = time.perf_counter()
r = Solution(big)
for _ in range(500):
    tg = random.choice(big)
    assert big[r.pick(tg)] == tg
t_res = time.perf_counter() - t
print("Cách 2: 10^4 pick xong dưới 2 giây:", t_map < 2.0)
print("Cách 3: 500 pick (n = 20000) xong dưới 10 giây:", t_res < 10.0)
assert t_map < 2.0 and t_res < 10.0

Cách 2: 10^4 pick xong dưới 2 giây: True
Cách 3: 500 pick (n = 20000) xong dưới 10 giây: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: thay đáp án với xác suất cố định `1/2`

Nhiều bạn nghĩ "mỗi ứng viên mới: tung đồng xu, ngửa thì thay" — nhưng như vậy ứng viên cuối có xác suất `1/2`, ứng viên đầu chỉ còn `1/2^k`...

In [25]:
random.seed(398)
cnt = sample_counts(WrongHalf, [1, 2, 3, 3, 3], 3, 30000)
print("sai (xu 1/2): ", [cnt[i] for i in (2, 3, 4)], "kỳ vọng ~10000 mỗi chỉ số")
random.seed(398)
cnt = sample_counts(Solution, [1, 2, 3, 3, 3], 3, 30000)
print("đúng (1/n):   ", [cnt[i] for i in (2, 3, 4)])

sai (xu 1/2):  [7399, 7555, 15046] kỳ vọng ~10000 mỗi chỉ số
đúng (1/n):    [9940, 9973, 10087]


**Vì sao sai:** với `WrongHalf`, ứng viên thứ nhất luôn được nhận (do `ans == -1`), thứ hai thay với xác suất `1/2`, thứ ba lại `1/2`. Xác suất cuối: `1/4, 1/4, 1/2`. **Cách sửa:** xác suất thay ở ứng viên thứ `n` phải là **`1/n`** — `random.randint(1, n) == n`.

### Lỗi 2: dùng `random.randint(0, n)` (lệch một đơn vị) hoặc `random.randrange(n)` sai mốc

`randint(a, b)` **gồm cả hai đầu**. `randint(0, n)` có `n + 1` giá trị nên xác suất `x == n` chỉ là `1/(n+1)`.

In [26]:
class WrongOffByOne:
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        n, ans = 0, -1
        for i, v in enumerate(self.nums):
            if v == target:
                n += 1
                if random.randint(0, n) == n or ans == -1:      # SAI: randint(0, n) có n+1 giá trị
                    ans = i
        return ans


random.seed(398)
cnt = sample_counts(WrongOffByOne, [1, 2, 3, 3, 3], 3, 30000)
counts = [cnt[i] for i in (2, 3, 4)]
print("sai:", counts, "chi2 =", round(chi_square(counts), 1), "| lệch?", not looks_uniform(counts))
assert not looks_uniform(counts)

sai: [14971, 7581, 7448] chi2 = 3707.5 | lệch? True


**Vì sao sai:** ứng viên thứ `n` bị nhận với xác suất `1/(n+1)` (thấp hơn `1/n`), ứng viên đầu được "bảo hộ" dễ giữ lâu hơn, nên phân phối nghiêng về các chỉ số đầu. **Cách sửa:** `randint(1, n) == n` (hoặc `randrange(n) == 0`).

### Lỗi 3: đếm `n` sai chỗ (đếm mọi phần tử thay vì chỉ các ứng viên khớp)

In [27]:
class WrongCountAll:
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        ans = -1
        for i, v in enumerate(self.nums):
            if v == target:
                if random.randint(1, i + 1) == i + 1 or ans == -1:    # SAI: dùng i + 1 thay vì số ứng viên đã gặp
                    ans = i
        return ans


random.seed(398)
cnt = sample_counts(WrongCountAll, [1, 2, 3, 3, 3], 3, 30000)
counts = [cnt[i] for i in (2, 3, 4)]
print("sai:", counts, "chi2 =", round(chi_square(counts), 1), "| lệch?", not looks_uniform(counts))
assert not looks_uniform(counts)

sai: [18015, 6019, 5966] chi2 = 9636.2 | lệch? True


**Vì sao sai:** `n` phải là số ứng viên **khớp** đã gặp. Dùng vị trí `i + 1` khiến xác suất thay `1/(i+1)` nhỏ hơn mức cần, nghiêng về ứng viên sớm. **Cách sửa:** biến đếm riêng `n += 1` chỉ khi `nums[i] == target`.

### Lỗi 4: dùng `random.shuffle` hoặc xáo trộn mảng gốc

Xáo `nums` rồi lấy `index` đầu tiên sẽ làm **chỉ số trả về không còn là chỉ số trong mảng gốc**. Nếu xáo một bản sao các **chỉ số**, vẫn đúng nhưng tốn `O(k)` bộ nhớ — không tốt hơn Cách 1.

### Lỗi 5: quên `import random` khi nộp

Trên LeetCode Python hay có sẵn một số mô-đun nhưng **đừng dựa vào đó**; tự viết `import random` ở đầu để tránh `NameError` khi chạy local.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng reservoir sampling:** "chọn ngẫu nhiên đều trong một luồng/danh sách **không biết trước độ dài**", "**O(1)** bộ nhớ", "chỉ duyệt một lần".
- **Công thức vàng:** phần tử thứ `n` được giữ với xác suất `1/n`; chứng minh bằng tích thu gọn.
- **Chọn `k` phần tử:** phần tử thứ `n` (khi `n > k`) vào bể với xác suất `k/n`, thay một phần tử ngẫu nhiên trong bể.
- **Khi cần `pick` nhanh nhiều lần,** đổi bộ nhớ lấy thời gian: `dict` hoặc sắp xếp + `bisect`.
- **Kiểm thử ngẫu nhiên:** (1) cố định `seed`; (2) kiểm tính hợp lệ; (3) thống kê chi-bình-phương; (4) **kiểm chính test** bằng cách cho nó thấy cài đặt sai bị bác bỏ.
- Với test thống kê, đừng đặt ngưỡng quá chặt (dễ "đỏ giả"); mức `0.1%` cộng seed cố định là an toàn.

## 15. Biến thể

1. **Chọn ngẫu nhiên có trọng số** (LeetCode 528): dùng tổng tiền tố + `bisect`.
2. **Reservoir sampling với `k` phần tử** (chọn `k` chỉ số khác nhau).
3. **Random priority:** gán mỗi ứng viên một số thực ngẫu nhiên `U(0, 1)`, lấy ứng viên có số **lớn nhất** — cũng đều (mỗi ứng viên có cơ hội ngang nhau là "lớn nhất").
4. **Shuffle an Array** (384), **Linked List Random Node** (382): cùng họ ngẫu nhiên đều.

In [28]:
class SolutionPriority:
    """Mỗi ứng viên nhận một số ngẫu nhiên; chọn ứng viên có số lớn nhất."""
    def __init__(self, nums: List[int]):
        self.nums = nums

    def pick(self, target: int) -> int:
        best_key, ans = -1.0, -1
        for i, v in enumerate(self.nums):
            if v == target:
                key = random.random()
                if key > best_key:
                    best_key, ans = key, i
        return ans


random.seed(398)
cnt = sample_counts(SolutionPriority, [1, 2, 3, 3, 3], 3, 30000)
counts = [cnt[i] for i in (2, 3, 4)]
print("random priority:", counts, "chi2 =", round(chi_square(counts), 2), "| đều?", looks_uniform(counts))
assert looks_uniform(counts)


def reservoir_k(stream, k):
    """Chọn k phần tử đều từ luồng (thuật toán R)."""
    bag = []
    for n, x in enumerate(stream, start=1):
        if n <= k:
            bag.append(x)
        else:
            j = random.randint(1, n)
            if j <= k:
                bag[j - 1] = x
    return bag


random.seed(1)
print(reservoir_k(range(10), 3))

random priority: [10009, 10040, 9951] chi2 = 0.41 | đều? True
[5, 7, 6]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét + danh sách | tạo danh sách ứng viên mỗi lần | `pick: O(n)` | `O(k)` mỗi lần | hiểu đề, `n` nhỏ |
| 2. Bảng băm | gom sẵn `giá trị → chỉ số` | `init O(n)`, `pick O(1)` | `O(n)` | nhiều lần `pick`, bộ nhớ thoải mái |
| 3. Reservoir | thay đáp án với xác suất `1/n` | `pick: O(n)` | **`O(1)`** | bộ nhớ hạn chế; luồng không biết độ dài; **nộp bài** |
| 4. Sắp xếp + bisect | đoạn ứng viên liền nhau, `randrange` | `init O(n log n)`, `pick O(log n)` | `O(n)` | cần `pick` nhanh và truy vấn theo khoảng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `random.randint(1, 1)`; (b) số phần tử của `pos[3]` khi `nums = [3, 1, 3, 3]`; (c) `bisect.bisect_left([1, 3, 3, 5], 3)` và `bisect.bisect_right([1, 3, 3, 5], 3)`; (d) với `k = 4` ứng viên, xác suất mỗi chỉ số là cuối cùng.

**Bài 2 (tính tay).** Với `k = 4` ứng viên, tính xác suất ứng viên thứ `2` là đáp án cuối bằng tích `(1/2) * (2/3) * (3/4)` và so với `1/4`.

**Bài 3 (điền TODO).** Hoàn thành lớp `SolutionTodo` theo Cách 2 (bảng băm).

In [29]:
class SolutionTodo:
    def __init__(self, nums):
        # TODO 1: tạo dict self.pos: giá trị -> danh sách chỉ số (dùng defaultdict(list) hoặc setdefault)
        # TODO 2: duyệt enumerate(nums) và append chỉ số vào pos[giá trị]
        pass

    def pick(self, target):
        # TODO 3: chọn ngẫu nhiên đều một chỉ số trong self.pos[target] bằng random.choice
        return None

# KHÔNG tạo đối tượng/gọi pick ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `SolutionRes2` theo Cách 3 (reservoir) **không nhìn lại** mục 7, rồi kiểm tra bằng `chi_square` trên `[4, 4, 1, 4, 2, 4]`, `target = 4`.

**Bài 5 (biến thể / mini challenge).** Viết `pick_two(nums, target)` trả về **hai chỉ số khác nhau** chọn đều trong các chỉ số của `target` (reservoir với `k = 2`). Kiểm tra thống kê: với 4 ứng viên có `C(4,2) = 6` cặp, mỗi cặp phải xuất hiện đều.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`; `3`; `1` và `3`; `1/4`. Bài 3: nhìn mục 3.2 để dựng `pos`.
- **Mức 2 (vừa):** Bài 2: `(1/2)*(2/3)*(3/4) = 6/24 = 1/4`. Bài 4: `n = 0`; khi khớp `n += 1`, `if random.randint(1, n) == n: ans = i`.
- **Mức 3 (sâu):** Bài 5: dùng `reservoir_k` ở mục 15 trên danh sách các chỉ số của `target` với `k = 2`; đếm bằng `Counter(tuple(sorted(pair)))`; kiểm bằng `looks_uniform` (6 cặp, mức tới hạn `20.52`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert` (các kiểm tra thống kê dùng seed cố định).

In [30]:
# Bài 1
assert random.randint(1, 1) == 1
pos3 = defaultdict(list)
for i, v in enumerate([3, 1, 3, 3]):
    pos3[v].append(i)
assert len(pos3[3]) == 3
assert bisect.bisect_left([1, 3, 3, 5], 3) == 1 and bisect.bisect_right([1, 3, 3, 5], 3) == 3
assert final_probability(2, 4) == Fraction(1, 4)

# Bài 2
assert Fraction(1, 2) * Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 4)


# Bài 3
class SolutionTodoAns:
    def __init__(self, nums):
        self.pos = defaultdict(list)
        for i, v in enumerate(nums):
            self.pos[v].append(i)

    def pick(self, target):
        return random.choice(self.pos[target])


random.seed(31)
c3 = Counter(SolutionTodoAns([1, 2, 3, 3, 3]).pick(3) for _ in range(30000))
assert looks_uniform([c3[2], c3[3], c3[4]])


# Bài 4
class SolutionRes2:
    def __init__(self, nums):
        self.nums = nums

    def pick(self, target):
        n, ans = 0, -1
        for i, v in enumerate(self.nums):
            if v == target:
                n += 1
                if random.randint(1, n) == n:
                    ans = i
        return ans


random.seed(41)
arr4 = [4, 4, 1, 4, 2, 4]
c4 = Counter(SolutionRes2(arr4).pick(4) for _ in range(40000))
assert set(c4) == {0, 1, 3, 5} and looks_uniform([c4[i] for i in (0, 1, 3, 5)])


# Bài 5
def pick_two(nums, target):
    positions = [i for i, v in enumerate(nums) if v == target]
    return tuple(sorted(reservoir_k(positions, 2)))


random.seed(51)
arr5 = [9, 7, 9, 9, 1, 9]
c5 = Counter(pick_two(arr5, 9) for _ in range(60000))
pairs = sorted(c5)
assert len(pairs) == 6 and all(a != b for a, b in pairs)
assert chi_square([c5[p] for p in pairs]) < CHI2_CRIT[6]
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (reservoir sampling, `O(1)` bộ nhớ)

```python
import random

class Solution:

    def __init__(self, nums: List[int]):
        self.nums = nums

    def pick(self, target: int) -> int:
        n = 0
        ans = 0
        for i, v in enumerate(self.nums):
            if v == target:
                n += 1                              # ứng viên thứ n
                if random.randint(1, n) == n:       # xác suất 1/n
                    ans = i
        return ans
```

**Checklist**

- [ ] Đã `import random`?
- [ ] `n` chỉ đếm các phần tử **khớp** `target` (không phải vị trí `i`)?
- [ ] Xác suất thay ở ứng viên thứ `n` là **`1/n`** (`randint(1, n) == n`), không phải `1/2` hay `1/(n+1)`?
- [ ] Hiểu tích thu gọn `(1/i) * (i/(i+1)) * ... * ((k-1)/k) = 1/k`?
- [ ] Chọn đúng đánh đổi: bảng băm (nhanh, tốn bộ nhớ) hay reservoir (chậm hơn, `O(1)` bộ nhớ)?
- [ ] Test bằng `seed` cố định + chi-bình-phương, và có thử một cài đặt sai để chắc test bắt được lỗi?